# AML Risk Prioritization — Model Development

**Business question:** At a fixed review capacity, how many positive *synthetic* labels can a risk-ranked queue capture?

This notebook documents one final modeling and evaluation workflow. The exact runnable implementation is `src/run_canonical.py`. We separate model ranking from operational decisions: the model scores account-days, and a review-capacity policy decides which records enter the queue.

> These are synthetic AML benchmark labels, not confirmed criminal activity or production performance.

## 1. Data contract and temporal holdout

Each observation is one account-day. The data has no personal customer records. Training uses **September 1–6, 2022**, validation uses **September 7–8**, and the untouched test uses **September 9–10**.

We use only training labels to set class weights and do not include `account_id`, `activity_date`, or the target among model features. The validation queue is frozen before test evaluation.

In [ ]:
from pathlib import Path
import os, pandas as pd
input_dir = Path(os.environ.get("AML_INPUT_DIR", "data/local"))
df = pd.read_csv(input_dir / "aml_modeling_dev.tsv", sep="\t")
df["activity_date"] = pd.to_datetime(df["activity_date"])
assert not df[["account_id", "activity_date"]].duplicated().any()
print("Rows:", len(df), "Dates:", df.activity_date.min(), "to", df.activity_date.max())
print("Positive label rate:", df.has_laundering_tx.mean())

## 2. Features and leakage control

The model uses nine predictors. Six skewed measures receive `log1p`; three ratios/diversity features remain untransformed. Missing/nonfinite features cause a validation error rather than silently entering the model. The date is used to split observations, not as a predictive input.

In [ ]:
import numpy as np
log_features = ["transaction_count", "total_sent", "total_received", "avg_transaction_amount", "max_transaction_amount", "unique_banks"]
raw_features = ["outgoing_ratio", "currency_diversity", "rapid_transaction_ratio"]
def make_features(data):
    x = pd.DataFrame(index=data.index)
    for name in log_features:
        x["log_" + name] = np.log1p(data[name])
    for name in raw_features:
        x[name] = data[name]
    assert np.isfinite(x.to_numpy()).all()
    return x
train = df[df.activity_date.between("2022-09-01", "2022-09-06")]
validation = df[df.activity_date.between("2022-09-07", "2022-09-08")]
test = df[df.activity_date.between("2022-09-09", "2022-09-10")]
X_train, X_validation, X_test = [make_features(d) for d in (train, validation, test)]
print("Train / validation / test:", len(train), len(validation), len(test))

## 3. Baseline and final model

A logistic regression model is a useful interpretable baseline; the final selected challenger is XGBoost. Its hyperparameters were selected during validation-stage development, **not by searching on the test set**. The canonical pipeline below fits the frozen final configuration once.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
y_train = train["has_laundering_tx"]
y_validation = validation["has_laundering_tx"]
baseline = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
baseline.fit(X_train, y_train)
print("Logistic validation PR-AUC:", round(average_precision_score(y_validation, baseline.predict_proba(X_validation)[:, 1]), 4))

In [ ]:
from xgboost import XGBClassifier
weight = (y_train.eq(0).sum() / y_train.eq(1).sum())
model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", eval_metric="aucpr", scale_pos_weight=weight, random_state=42, n_jobs=2)
model.fit(X_train, y_train)
print("XGBoost validation PR-AUC:", round(average_precision_score(y_validation, model.predict_proba(X_validation)[:, 1]), 4))

## 4. Frozen review capacity and held-out evaluation

Choose a 2,500-case validation workload, translate it to the equivalent test fraction, then rank the held-out test scores. A stable sort resolves ties without random tie-breaking. Precision is the proportion of selected account-days with positive labels; recall is the proportion of all positive labels captured; lift divides selected precision by the test positive-label rate.

In [ ]:
from sklearn.metrics import roc_auc_score
y_test = test["has_laundering_tx"].to_numpy()
prob = model.predict_proba(X_test)[:, 1]
capacity = round(len(test) * 2500 / len(validation))
order = np.argsort(-prob, kind="stable")
tp = int(y_test[order[:capacity]].sum())
precision, recall = tp / capacity, tp / int(y_test.sum())
lift = precision / float(y_test.mean())
print({"account_days": len(test), "review_capacity": capacity, "selected_positives": tp, "precision": precision, "recall": recall, "lift": lift, "ROC_AUC": roc_auc_score(y_test, prob), "PR_AUC": average_precision_score(y_test, prob)})

## 5. Export and reproducibility

Use the same tested pipeline to save a trained model, ranked prediction CSV, JSON metrics, and the capacity curve that powers the interactive website. The output folder is excluded from Git. Compare the generated values against `reports/metrics.json` and the published website before releasing changes.

In [ ]:
%run src/run_canonical.py

## 6. Interpretation and limitations

At the frozen capacity of **2,747 account-days**, the final held-out run selects **389** positive synthetic labels from **1,499** total positives (precision **14.16%**, recall **25.95%**, lift **46.73×**). The model's ROC-AUC is **0.8523** and PR-AUC is **0.1214**. The short holdout period, synthetic labels, extreme imbalance, and unknown real-world review costs prevent deployment claims. Model scores should not be described as calibrated probabilities.

**For reviewers:** `reports/MODEL_EVALUATION.md` provides the evaluation protocol and `src/run_canonical.py` is the reproducible implementation.